# Simple: Uniform Numeric Noise with PAMOLA.CORE

**Goal**: Learn uniform noise addition basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply uniform noise to numeric fields using execute()
- Compare before/after results
- Understand privacy-utility tradeoffs

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/noise/
        └── 01_uniform_numeric_noise_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.noise.uniform_numeric_op import (
        UniformNumericNoiseOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with numeric sensor readings (temperature, humidity, pressure)

In [ ]:
# Define path to sample data (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

# Check if file exists
if not data_path.exists():
    print(f"⚠️  File not found: {data_path}")
    print("Creating sample data...\n")
    
    # Create directory
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with realistic sensor readings
    np.random.seed(42)
    n_records = 50
    
    sample_data = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'timestamp': pd.date_range('2024-01-01', periods=n_records, freq='H'),
        'temperature': np.round(np.random.normal(22.5, 2.5, n_records), 2),  # °C
        'humidity': np.round(np.random.normal(55, 10, n_records), 2),        # %
        'pressure': np.round(np.random.normal(1013.25, 5, n_records), 2),   # hPa
        'sensor_id': np.random.choice(['SENSOR_A', 'SENSOR_B', 'SENSOR_C'], n_records),
        'location': np.random.choice(['Room_101', 'Room_102', 'Room_103'], n_records)
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created at: {data_path}\n")

# Load data using pandas
df = read_csv(data_path)

print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}, std={df[col].std():.2f}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field name** for noise addition

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "years_of_experience"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "years_of_experience"  # Field to add noise - CUSTOMIZE THIS!
    }

# Get config
kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists in dataset
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

# Validate field is numeric
if not pd.api.types.is_numeric_dtype(df[field_name]):
    raise ValueError(
        f"❌ Field '{field_name}' is not numeric (dtype: {df[field_name].dtype})!\n"
        f"Uniform noise requires numeric fields."
    )

print(f"✓ Field to add noise: '{field_name}'")
print(f"  Data type: {df[field_name].dtype}")
print(f"  Range: {df[field_name].min():.2f} - {df[field_name].max():.2f}")
print(f"  Mean: {df[field_name].mean():.2f}")
print(f"  Std Dev: {df[field_name].std():.2f}")
print(f"  Unique values: {df[field_name].nunique()}")

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_noise_001",
    task_type="uniform_noise",
    description=f"Uniform noise addition to '{field_name}' field.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config kwargs (without field_name for execute)
execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create UniformNumericNoiseOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_name='{field_name}'`: Field to add noise to
- `noise_range=1.0`: Uniform noise ±1.0 (symmetric range)
- `noise_type='additive'`: Add noise to values (vs multiplicative)
- `output_min=None`: No minimum bound constraint
- `output_max=None`: No maximum bound constraint
- `preserve_zero=False`: Add noise to zero values too
- `use_secure_random=True`: Use cryptographically secure RNG
- `mode='REPLACE'`: Modify original field (or 'ENRICH' for new field)
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = UniformNumericNoiseOperation(
    # Core parameters
    field_name=field_name,               # From config
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Noise parameters
    noise_range=1.0,                     # ±1.0 uniform noise
    noise_type='additive',               # Options: 'additive', 'multiplicative'
    
    # Bounds and constraints
    output_min=None,                     # No minimum bound
    output_max=None,                     # No maximum bound
    preserve_zero=False,                 # Add noise to zeros
    
    # Integer handling
    round_to_integer=None,               # Auto-detect from data type
    
    # Statistical parameters
    scale_by_std=False,                  # Don't scale by std dev
    scale_factor=1.0,                    # Noise multiplier
    
    # Randomization
    random_seed=None,                    # No seed (different each run)
    use_secure_random=True,              # Use secure RNG
    
    # Processing settings
    use_vectorization=False,             # Disable for small data
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,         # Create visualization artifacts
    save_output=True,                    # Save results to files
    force_recalculation=False            # Use cache when available
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Noise type:       {operation.noise_type}")
print(f"  Noise range:      ±{operation.noise_range}")
print(f"  Secure random:    {operation.use_secure_random}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing uniform noise addition...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the noisy output from task_dir
- Compare with original data
- Review metrics and artifacts

**Generated artifacts:**
- Noisy CSV in output/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records with before/after
    print("\n🔍 Sample Records (Original → Noisy):")
    print("-" * 80)
    print(f"{'Record':>6} {'Original':>12} {'Noisy':>12} {'Noise':>12}")
    print("-" * 80)
    
    for i in range(min(15, len(df))):
        original = df[field_name].iloc[i]
        noisy = result_df[field_name].iloc[i]
        noise = noisy - original
        print(f"  {i+1:4d}. {original:11.2f} → {noisy:11.2f}  ({noise:+.2f})")
    
    # Calculate noise statistics
    actual_noise = result_df[field_name] - df[field_name]
    
    print("\n" + "=" * 80)
    print("✨ NOISE ANALYSIS")
    print("=" * 80)
    print(f"\n📊 Original Statistics:")
    print(f"  Mean:     {df[field_name].mean():8.2f}")
    print(f"  Std Dev:  {df[field_name].std():8.2f}")
    print(f"  Min:      {df[field_name].min():8.2f}")
    print(f"  Max:      {df[field_name].max():8.2f}")
    
    print(f"\n📊 Noisy Statistics:")
    print(f"  Mean:     {result_df[field_name].mean():8.2f}")
    print(f"  Std Dev:  {result_df[field_name].std():8.2f}")
    print(f"  Min:      {result_df[field_name].min():8.2f}")
    print(f"  Max:      {result_df[field_name].max():8.2f}")
    
    print(f"\n📊 Noise Statistics:")
    print(f"  Mean:     {actual_noise.mean():8.2f}")
    print(f"  Std Dev:  {actual_noise.std():8.2f}")
    print(f"  Min:      {actual_noise.min():8.2f}")
    print(f"  Max:      {actual_noise.max():8.2f}")
    print(f"  Range:    [{actual_noise.min():.2f}, {actual_noise.max():.2f}]")
    
    # Signal-to-Noise Ratio
    snr = df[field_name].std() / actual_noise.std() if actual_noise.std() > 0 else float('inf')
    print(f"\n📈 Signal-to-Noise Ratio (SNR): {snr:.2f}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Uniform noise provides differential privacy by adding controlled randomness!")
    print(f"   Higher SNR = Better utility (noise is small relative to signal)")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Noisy CSV
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

# List all subdirectories and files
for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:  # Show first 5 files
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")
        if len(files) > 5:
            print(f"   ... and {len(files) - 5} more files")
        print()

print("=" * 80)
print("\n✅ All artifacts saved successfully!")
print(f"\n💡 TIP: Navigate to {task_dir.absolute()} to inspect files manually")

## Step 7: Detailed Artifact Review

**How to use:**
- Review metrics JSON with noise statistics
- Inspect output data distribution
- View visualizations

**Note:** Always shows the NEWEST files from the latest operation run.

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. Display Metrics JSON (NEWEST FILE)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types inside IF
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display effectiveness metrics
            if 'effectiveness' in metrics:
                print("\n📈 EFFECTIVENESS:")
                eff = metrics['effectiveness']
                print(f"   Total Records: {eff.get('total_records', 0):,}")
                print(f"   Original Unique: {eff.get('original_unique', 0):,}")
                print(f"   Anonymized Unique: {eff.get('anonymized_unique', 0):,}")
                print(f"   Effectiveness Ratio: {eff.get('effectiveness_ratio', 0):.4f}")
                print(f"   Null Increase: {eff.get('null_increase', 0):.2%}")
            
            # Display performance metrics
            if 'duration_seconds' in metrics:
                print("\n⚡ PERFORMANCE:")
                print(f"   Duration: {metrics.get('duration_seconds', 0):.4f}s")
                print(f"   Records Processed: {metrics.get('records_processed', 0):,}")
                rps = metrics.get('records_per_second', 0)
                print(f"   Records/Second: {rps:,.0f}")
                print(f"   Chunk Count: {metrics.get('chunk_count', 1)}")
            
            # Display noise configuration
            if 'noise_type' in metrics:
                print("\n🎲 NOISE CONFIGURATION:")
                print(f"   Noise Type: {metrics.get('noise_type', 'N/A').title()}")
                noise_range = metrics.get('noise_range', 'N/A')
                if isinstance(noise_range, (list, tuple)):
                    print(f"   Noise Range: [{noise_range[0]}, {noise_range[1]}]")
                else:
                    print(f"   Noise Range: ±{noise_range}")
                print(f"   Secure Random: {'Yes' if metrics.get('secure_random', False) else 'No'}")
                
                if metrics.get('preserved_zeros') is not None:
                    print(f"   Preserved Zeros: {metrics.get('preserved_zeros', 0)} records")
            
            # Display actual noise statistics
            if 'actual_noise_mean' in metrics:
                print("\n📊 ACTUAL NOISE STATISTICS:")
                print(f"   Mean: {metrics.get('actual_noise_mean', 0):+.6f}")
                print(f"   Std Dev: {metrics.get('actual_noise_std', 0):.6f}")
                print(f"   Min: {metrics.get('actual_noise_min', 0):+.6f}")
                print(f"   Max: {metrics.get('actual_noise_max', 0):+.6f}")
                
                # Signal-to-Noise Ratio
                snr = metrics.get('signal_to_noise_ratio', 0)
                print(f"\n   📈 Signal-to-Noise Ratio: {snr:.2f}")
                if snr == float('inf'):
                    print("      ✓ Perfect SNR (no noise variation)")
                elif snr > 10:
                    print("      ✓ High utility preserved (SNR > 10)")
                elif snr > 5:
                    print("      ⚠️ Moderate utility (5 < SNR < 10)")
                else:
                    print("      ❌ Low utility (SNR < 5)")
            
            # Display bounds information
            if 'values_at_bounds' in metrics:
                bounds = metrics['values_at_bounds']
                at_min = bounds.get('at_min', 0)
                at_max = bounds.get('at_max', 0)
                
                if at_min > 0 or at_max > 0:
                    print("\n🔒 BOUNDS ENFORCEMENT:")
                    if at_min > 0:
                        print(f"   At Minimum Bound: {at_min:,} records")
                    if at_max > 0:
                        print(f"   At Maximum Bound: {at_max:,} records")
            
            # Display noise impact metrics
            if 'noise_impact' in metrics:
                impact = metrics['noise_impact']
                print("\n📈 NOISE IMPACT ANALYSIS:")
                
                if 'rmse' in impact:
                    print(f"   Root Mean Squared Error: {impact.get('rmse', 0):.6f}")
                if 'mae' in impact:
                    print(f"   Mean Absolute Error: {impact.get('mae', 0):.6f}")
                if 'max_absolute_error' in impact:
                    print(f"   Max Absolute Error: {impact.get('max_absolute_error', 0):.6f}")
                
                if 'correlation' in impact:
                    corr = impact.get('correlation', 0)
                    print(f"\n   🔗 Correlation: {corr:.6f}")
                    if corr > 0.95:
                        print("      ✓ Excellent correlation (> 0.95)")
                    elif corr > 0.85:
                        print("      ✓ Good correlation (> 0.85)")
                    elif corr > 0.7:
                        print("      ⚠️ Moderate correlation (> 0.7)")
                    else:
                        print("      ❌ Weak correlation (< 0.7)")
                
                if 'rank_correlation' in impact:
                    print(f"   Rank Correlation: {impact.get('rank_correlation', 0):.6f}")
                
                if 'snr' in impact:
                    print(f"   SNR (from impact): {impact.get('snr', 0):.2f} dB")
                
                if 'relative_error' in impact:
                    rel_err = impact.get('relative_error', 0)
                    if rel_err < 1e6:  # Only show if reasonable
                        print(f"   Relative Error: {rel_err:.2%}")
            
            # Display distribution preservation
            if 'distribution_preservation' in metrics:
                dist = metrics['distribution_preservation']
                print("\n📊 DISTRIBUTION PRESERVATION:")
                
                if 'ks_statistic' in dist:
                    ks_stat = dist.get('ks_statistic', 0)
                    ks_pval = dist.get('ks_pvalue', 0)
                    print(f"   Kolmogorov-Smirnov Test:")
                    print(f"      Statistic: {ks_stat:.6f}")
                    print(f"      P-Value: {ks_pval:.6f}")
                    if ks_pval > 0.05:
                        print(f"      ✓ Distributions are similar (p > 0.05)")
                    else:
                        print(f"      ⚠️ Distributions differ (p < 0.05)")
                
                if 'wasserstein_distance' in dist:
                    print(f"\n   Wasserstein Distance: {dist.get('wasserstein_distance', 0):.6f}")
                
                if 'mean_shift' in dist:
                    print(f"   Mean Shift: {dist.get('mean_shift', 0):+.6f}")
                if 'std_ratio' in dist:
                    print(f"   Std Dev Ratio: {dist.get('std_ratio', 0):.6f}")
                
                if 'skewness_diff' in dist:
                    print(f"   Skewness Difference: {dist.get('skewness_diff', 0):+.6f}")
                if 'kurtosis_diff' in dist:
                    print(f"   Kurtosis Difference: {dist.get('kurtosis_diff', 0):+.6f}")
                
                if 'percentile_shifts' in dist:
                    pct_shifts = dist['percentile_shifts']
                    print(f"\n   Percentile Shifts:")
                    for pct_name, shift in pct_shifts.items():
                        print(f"      {pct_name.upper()}: {shift:+.6f}")
            
            # Display uniformity analysis
            if 'uniformity_analysis' in metrics:
                uniformity = metrics['uniformity_analysis']
                print("\n🎯 UNIFORMITY ANALYSIS:")
                
                if 'uniformity_test' in uniformity:
                    test = uniformity['uniformity_test']
                    print(f"   Chi-Square Test:")
                    print(f"      Statistic: {test.get('chi2_statistic', 0):.2f}")
                    print(f"      P-Value: {test.get('chi2_p_value', 0):.6f}")
                    is_uniform_chi2 = test.get('is_uniform_chi2', False)
                    print(f"      Is Uniform: {'Yes' if is_uniform_chi2 else 'No'}")
                    
                    print(f"\n   Kolmogorov-Smirnov Test:")
                    print(f"      Statistic: {test.get('ks_statistic', 0):.6f}")
                    print(f"      P-Value: {test.get('ks_p_value', 0):.6f}")
                    is_uniform_ks = test.get('is_uniform_ks', False)
                    print(f"      Is Uniform: {'Yes' if is_uniform_ks else 'No'}")
                
                if 'actual_range' in uniformity:
                    range_info = uniformity['actual_range']
                    print(f"\n   Noise Range:")
                    print(f"      Actual: [{range_info.get('min', 0):.2f}, {range_info.get('max', 0):.2f}]")
                    print(f"      Expected: [{range_info.get('expected_min', 0):.2f}, {range_info.get('expected_max', 0):.2f}]")
                    within = range_info.get('within_bounds', False)
                    print(f"      Within Bounds: {'Yes ✓' if within else 'No ❌'}")
                
                if 'distribution_metrics' in uniformity:
                    dist_metrics = uniformity['distribution_metrics']
                    print(f"\n   Distribution Statistics:")
                    print(f"      Mean: {dist_metrics.get('mean', 0):+.6f} (expected: {dist_metrics.get('expected_mean', 0):.6f})")
                    print(f"      Std: {dist_metrics.get('std', 0):.6f} (expected: {dist_metrics.get('expected_std', 0):.6f})")
                    print(f"      Skewness: {dist_metrics.get('skewness', 0):+.6f}")
                    print(f"      Kurtosis: {dist_metrics.get('kurtosis', 0):+.6f}")
            
            # Display noise effectiveness
            if 'noise_effectiveness' in metrics:
                effectiveness = metrics['noise_effectiveness']
                print("\n✨ NOISE EFFECTIVENESS:")
                
                is_effective = effectiveness.get('effective', False)
                print(f"   Overall Effective: {'Yes ✓' if is_effective else 'No ❌'}")
                
                print(f"   Privacy Metric: {effectiveness.get('privacy_metric', 'N/A').upper()}")
                print(f"   Actual Value: {effectiveness.get('actual_value', 0):.2f}")
                print(f"   Target Value: {effectiveness.get('target_value', 0):.2f}")
                
                utility_score = effectiveness.get('utility_score', 0)
                print(f"\n   📊 Utility Score: {utility_score:.2%}")
                
                if 'utility_components' in effectiveness:
                    components = effectiveness['utility_components']
                    print(f"\n   Utility Components:")
                    print(f"      Correlation: {components.get('correlation', 0):.4f}")
                    rel_err = components.get('relative_error', 0)
                    if rel_err < 1:  # Only show if reasonable
                        print(f"      Relative Error: {rel_err:.6f}")
                    print(f"      Mean Preservation: {components.get('mean_preservation', 0):.4f}")
                    print(f"      Std Preservation: {components.get('std_preservation', 0):.4f}")
                    print(f"      Distribution Similarity: {components.get('distribution_similarity', 0):.4f}")
                
                if 'recommendations' in effectiveness:
                    recs = effectiveness['recommendations']
                    if recs:
                        print(f"\n   💡 Recommendations:")
                        for rec in recs:
                            print(f"      • {rec}")
            
            # Display processing statistics
            if 'total_records' in metrics:
                print("\n📊 PROCESSING STATISTICS:")
                print(f"   Total Records: {metrics.get('total_records', 0):,}")
                print(f"   Processed Records: {metrics.get('processed_records', 0):,}")
                print(f"   Filtered Records: {metrics.get('filtered_records', 0):,}")
                print(f"   Processing Rate: {metrics.get('processing_rate', 0):.1f}%")
            
            # Display memory optimization info
            if 'chunk_size_used' in metrics:
                print("\n💾 MEMORY OPTIMIZATION:")
                print(f"   Chunk Size Used: {metrics.get('chunk_size_used', 0):,}")
                adaptive = metrics.get('adaptive_chunk_size', False)
                print(f"   Adaptive Chunking: {'Enabled ✓' if adaptive else 'Disabled'}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. Display Output Data Comparison (NEWEST FILE)
print("\n\n2️⃣ OUTPUT DATA (Before/After Comparison):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            
            # Create detailed comparison
            if field_name in output_df.columns and field_name in df.columns:
                print("📊 Detailed Comparison (First 20 records):")
                print("-" * 80)
                print(f"{'ID':>4} {'Original':>12} {'Noisy':>12} {'Noise':>12} {'% Change':>10}")
                print("-" * 80)
                
                for i in range(min(20, len(df))):
                    orig = df[field_name].iloc[i]
                    noisy = output_df[field_name].iloc[i]
                    noise = noisy - orig
                    pct_change = (noise / orig * 100) if orig != 0 else 0
                    
                    print(f"{i+1:4d} {orig:12.2f} {noisy:12.2f} {noise:+12.2f} {pct_change:+9.1f}%")
                
                # Statistical summary
                actual_noise = output_df[field_name] - df[field_name]
                
                print("\n" + "=" * 80)
                print("📊 STATISTICAL SUMMARY")
                print("=" * 80)
                
                print("\n📈 Original Data:")
                print(f"   Count:    {len(df):8,}")
                print(f"   Mean:     {df[field_name].mean():12.4f}")
                print(f"   Std Dev:  {df[field_name].std():12.4f}")
                print(f"   Min:      {df[field_name].min():12.4f}")
                print(f"   Max:      {df[field_name].max():12.4f}")
                print(f"   Median:   {df[field_name].median():12.4f}")
                
                print("\n📊 Noisy Data:")
                print(f"   Count:    {len(output_df):8,}")
                print(f"   Mean:     {output_df[field_name].mean():12.4f}")
                print(f"   Std Dev:  {output_df[field_name].std():12.4f}")
                print(f"   Min:      {output_df[field_name].min():12.4f}")
                print(f"   Max:      {output_df[field_name].max():12.4f}")
                print(f"   Median:   {output_df[field_name].median():12.4f}")
                
                print("\n🎲 Noise Distribution:")
                print(f"   Mean:     {actual_noise.mean():12.4f}")
                print(f"   Std Dev:  {actual_noise.std():12.4f}")
                print(f"   Min:      {actual_noise.min():12.4f}")
                print(f"   Max:      {actual_noise.max():12.4f}")
                print(f"   Median:   {actual_noise.median():12.4f}")
                
                # Percentiles
                print("\n📊 Noise Percentiles:")
                percentiles = [5, 25, 50, 75, 95]
                for p in percentiles:
                    val = np.percentile(actual_noise, p)
                    print(f"   {p:2d}th:     {val:12.4f}")
                
                # Signal-to-Noise Ratio
                snr = df[field_name].std() / actual_noise.std() if actual_noise.std() > 0 else float('inf')
                print(f"\n📈 Signal-to-Noise Ratio: {snr:.2f}")
                if snr > 10:
                    print("   ✓ High utility preserved (SNR > 10)")
                elif snr > 5:
                    print("   ⚠️ Moderate utility (5 < SNR < 10)")
                else:
                    print("   ❌ Low utility (SNR < 5)")
                
                # Correlation
                corr = df[field_name].corr(output_df[field_name])
                print(f"\n🔗 Correlation: {corr:.4f}")
                if corr > 0.9:
                    print("   ✓ Strong correlation maintained")
                elif corr > 0.7:
                    print("   ⚠️ Moderate correlation")
                else:
                    print("   ❌ Weak correlation")
            else:
                print(f"⚠️  Field '{field_name}' not found in output or original data")
        
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. Display Visualizations (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            for i, viz_file in enumerate(latest_viz_batch, 1):
                viz_name = viz_file.stem.replace('_', ' ').title()
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV with numeric sensor data  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review noise statistics and artifacts

**Key takeaways:**
- `operation.execute()` handles end-to-end noise addition
- Field name is configurable via `create_config_kwargs()`
- Uniform noise adds controlled randomness for differential privacy
- SNR measures privacy-utility tradeoff (higher = better utility)
- Secure random generation prevents predictable patterns
- All artifacts saved in structured directories

**Privacy-Utility Balance:**
- **High SNR (>10)**: Strong utility, moderate privacy
- **Medium SNR (5-10)**: Balanced privacy-utility
- **Low SNR (<5)**: Strong privacy, reduced utility

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_uniform_numeric_noise_advanced.ipynb`** includes:
- Multiplicative noise mode
- Output bounds enforcement
- Zero value preservation
- Statistical scaling (scale_by_std)
- Integer rounding options
- Custom scale factors
- Reproducible results with seeds
- Performance optimization for large datasets

**Other noise operations:**
- 📗 **Gaussian Noise**: Normal distribution for differential privacy
- 📙 **Laplace Noise**: Standard differential privacy mechanism
- 📕 **Exponential Noise**: For non-negative values

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Noise Operations Guide](../../docs/noise_operations.md)
- 📊 [Differential Privacy Guide](../../docs/differential_privacy.md)